# Gevaert lab NSCLC segmentation workflow

Important: Launch Jupyter notebook from XNAT project level to run this notebook.

This notebook rewrites the original Gevaert lab demo to use the updated
`workflow_adapters.py` API, following the same pattern as `example-workflow.ipynb`.
It demonstrates how to:

* analyze an XNAT project for structural scans and associated segmentations,
* build per-session job descriptors with `populate_job_fields()`,
* generate per-job shell scripts with `workflow_to_batch()`,
* optionally upload job bundles to XNAT and launch Container Service jobs.


## 1. Mandatory user-settable variables

Edit this cell to set the main notebook variables.


In [3]:
from pathlib import Path

#set to True to regenerate project directory structure saved in local json file.
rebuild_directory_structure=False

# XNAT project label
project='NSCLC_RADIOMICS'

# Persistent workspace root path
root_dir=Path("/workspace/mmilchenko")


## 2. General initializations

Run this cell to set up the notebook environment.


In [4]:
import os
import sys
import json
import datetime
import yaml
import importlib

# nibabel 5.4.x currently pulls NumPy 2.x, which can break this notebook stack.
! pip install nibabel==5.1.0
! pip install cmdline_provenance

# Library with XNAT Jupyter workflow Python and shell scripts.
pymipl_path = os.path.abspath('../')
sys.path.append(pymipl_path)
sys.path.append(os.path.abspath(pymipl_path + '/xnat_workflow'))

# dicom_sort is part of pymipl. It analyzes XNAT projects for structural scans
# and segmentations.
from dicom_sort import *

# Derived variable initializations
local_workdir_path = root_dir / project
xnat_project_path = f'/data/projects/{project}/experiments'
directory_structure_file = local_workdir_path / "project_dir_structure.json"
xnat_structure_file = local_workdir_path / "xnat_structure.json"
scanlist_file = local_workdir_path / "scans.csv"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.0 MB/s eta 0:00:00ta 0:00:01


## 3. Create a list of structural scans and associated segmentations

Run the next cell to analyze the project structure and build the scan list.


In [5]:
# analyze_dir finds all structural scans and segmentations (DICOM RTSTRUCT
# and DICOM Segmentation Object) in the project. The results are saved into a
# JSON file for quick rerun.
if rebuild_directory_structure:
    os.makedirs(os.path.dirname(directory_structure_file), exist_ok=True)
    d = analyze_dir(xnat_project_path, directory_structure_file)
else:
    with open(directory_structure_file, 'r') as file:
        d = json.load(file)

# This writes out a human-readable list of structural scans with segmentations
# into a CSV file.
subjects, scans = reindex_to_structurals_and_segs(
    d,
    xnat_structure_file,
    scanlist_file
)
print(f'Number of structural scans: {len(scans)}')
print('First scan: ', scans[0])


Number of structural scans: 422
First scan:  {'Subject': 'LUNG1-093', 'Experiment': '04-13-2006-StudyID-NA-25111', 'StructScan': '0', 'StructScanSerDesc': '', 'SegScan1': '3', 'SegScan1_SerDesc': '', 'SegScan1_SOPClass': 'RTStruct', 'SegScan2': '300', 'SegScan2_SerDesc': 'Segmentation', 'SegScan2_SOPClass': 'Seg'}


## 4. Workflow initializations

This section configures the workflow using the updated adapter API.


In [9]:
import workflow_adapters as wa
import nibabel as nib

importlib.reload(wa)

# env_type controls where scripts are built and executed.
# jupyter: all processing runs in a single XNAT-Jupyter environment
# container: processing runs in dedicated containers, one per scan/session

#env_type='jupyter'
env_type='container'

# This is the label of the session resource where generated job scripts, logs,
# and output will be saved.
workflow_id='nsclc-segmentation-codebase-20260317'

# Command ID of the dedicated container command.
xnat_command_id=21

# Command wrapper name of the dedicated container command.
# Update this if your XNAT command uses a different wrapper name.
xnat_command_wrapper_id='xnat-ai-workflow-gevaert'

# User micromamba environment repository.
user_env_repo='gevaert'

# User source/resource directory.
user_src_repo='nsclc-segmentation-codebase-20260317'

if env_type == 'jupyter':
    global_vars = wa.init_global_vars(env_type, project,  workflow_id,
        g_user_env_repo=user_env_repo,
        g_user_src_repo=user_src_repo,
        g_local_workdir_path=local_workdir_path,
        g_pymipl_dir=root_dir / 'pymipl'
    )
elif env_type == 'container':
    global_vars = wa.init_global_vars(env_type, project, workflow_id)

global_vars


{'g_project': 'NSCLC_RADIOMICS',
 'g_workflow_id': 'nsclc-segmentation-codebase-20260317',
 'g_user_env_repo': 'NONE',
 'g_env_repo_dir': PosixPath('/opt/packages/user/env_repo'),
 'g_user_src_repo': 'NONE',
 'g_alg_repo_dir': PosixPath('/opt/packages/user/alg_repo'),
 'g_input_mount_path': PosixPath('/input'),
 'g_local_workdir_path': PosixPath('/workdir'),
 'g_pymipl_dir': PosixPath('/opt/packages/pymipl')}

## 5. Build per-session job descriptors and scripts

This cell uses the updated workflow API. It creates a job dictionary with
`populate_job_fields()`, appends workflow steps to `job['steps']`, writes
`job.yaml` and `job.sh`, and can optionally upload and launch jobs.


In [10]:
# List keys available from the scan table.
scans[0].keys()


dict_keys(['Subject', 'Experiment', 'StructScan', 'StructScanSerDesc', 'SegScan1', 'SegScan1_SerDesc', 'SegScan1_SOPClass', 'SegScan2', 'SegScan2_SerDesc', 'SegScan2_SOPClass'])

In [11]:
import workflow_adapters as wa
import importlib
importlib.reload(wa)

##################################################################
# Variables that control execution logic.

# Start and end positions in the CSV-derived scan list.
start_pos=2
end_pos=3

# Whether to upload the generated job script bundle to the session resource.
UpdateSessionResource=1

# Whether to run the container immediately after upload.
RunContainer=1

#################################################################
# Other initializations
wa.set_logger()
dt = datetime.datetime.now().strftime("%Y%m%d_%H%M")
batch_file = local_workdir_path / f"batch_{dt}.sh"
n = 0
xnat_interface = None
num_sessions = len(scans)

# Main loop over structural scans. This builds the per-scan script and,
# optionally, uploads and launches it.
for scan in scans:
    n = n + 1
    if n < start_pos or n > end_pos:
        continue

    ##########################################################################
    # Initialize job context with the updated API.
    job = wa.populate_job_fields(env_type, global_vars, workflow_id, scan)

    # Workflow-specific job variables.
    job['job_struct_path'] = job['job_scan_context'] / scan['StructScan'] / 'DICOM'

    ##########################################################################
    # Add workflow steps.

    step = {'step_title': "1. Run NSCLC tumor segmentation on structural scan"}
    step['step_command'] = (
        "micromamba run -p {g_env_repo_dir} python "
        "{g_alg_repo_dir}/run_segmentation.py "
        "--input {job_struct_path} "
        "--output-dir {job_workdir} "
        "--output-format nrrd "
        "--verbose"
    )
    job['steps'] += [step]

    step = {'step_title': "2. Convert structural scan to NIFTI"}
    step['step_command'] = (
        "micromamba run -n base python {g_pymipl_dir}/test_rt-utils.py "
        "{job_struct_path} {job_workdir}/ct"
    )
    job['steps'] += [step]

    step = {'step_title': "3. Move lesion mask to the job workdir root"}
    step['step_command'] = (
        "mv {job_workdir}/DICOM/lesion_mask.nii.gz "
        "{job_workdir}/lesion_mask.nii.gz"
    )
    job['steps'] += [step]

    step = {'step_title': "4. Generate QC image"}
    step['step_command'] = (
        "micromamba run -n base python {g_pymipl_dir}/slice_qc.py "
        "-o {job_workdir}/qc.png "
        "--mask {job_workdir}/lesion_mask.nii.gz "
        "{job_workdir}/ct_struct.nii"
    )
    job['steps'] += [step]

    step = {'step_title': "5. Convert output to DICOM RT"}
    step['step_command'] = (
        "micromamba run -n base python {g_pymipl_dir}/slice_qc.py "
        "-o {job_workdir}/qc.png "
        "--mask {job_workdir}/lesion_mask.nii.gz "
        "{job_workdir}/ct_struct.nii"
    )
    job['steps'] += [step]
   

    step = {'step_title': "6. Clean up intermediate files"}
    step['step_command'] = (
        "rm -r {job_workdir}/ct_struct.nii {job_workdir}/DICOM"
    )
    job['steps'] += [step]

    step = {'step_title': "6. Upload results to XNAT"}
    step['step_command'] = (
        "micromamba run -n base python "
        "{g_pymipl_dir}/xnat_workflow/sync-resource-with-xnat.py "
        "--level experiment "
        "--project {g_project} "
        "--subject {job_subject} "
        "--experiment {job_exp_label} "
        "--local_resource {job_workdir} "
        "--remote_resource {g_workflow_id} "
        "--create_hierarchy 1"
    )
    job['steps'] += [step]

    ##########################################################################
    # Write configuration and the final script.
    local_job_dir = local_workdir_path / 'jobs' / job['job_id']
    job_file_yaml = local_job_dir / 'job.yaml'
    job_file_sh = local_job_dir / 'job.sh'
    local_job_dir.mkdir(parents=True, exist_ok=True)

    with open(job_file_yaml, "w") as f:
        yaml.safe_dump(wa.paths_to_str(job), f, sort_keys=False)

    if env_type == 'jupyter':
        if not Path(batch_file).exists():
            with open(batch_file, 'w') as f:
                f.write('#!/bin/bash\n')
                f.write('export PATH="$PATH":/workspace/mmilchenko/bin\n')
                f.write('eval "$(micromamba shell hook --shell bash)"\n')

        wa.workflow_to_batch(job, global_vars, batch_file)
        print(f'Written {batch_file}')

    else:
        print(f'Written {job_file_sh}')
        Path(job_file_sh).write_text('')
        wa.workflow_to_batch(job, global_vars, job_file_sh)
        Path(job_file_sh).chmod(0o755)

        if UpdateSessionResource or RunContainer:
            if xnat_interface is None:
                xnat_interface = wa.get_xnat_interface(project)

        if UpdateSessionResource:
            print('Sending job bundle to XNAT resource')
            res1 = wa.resource_to_xnat(
                local_job_dir,
                workflow_id,
                project,
                job['job_subject'],
                job['job_exp_label'],
                xnat_interface
            )
            if res1 != 0:
                raise ConnectionError(
                    "Uploading job configuration to XNAT failed"
                )

        if RunContainer:
            print('Submitting job to Container Service')
            exp_id = (
                xnat_interface
                .select.project(project)
                .subject(job['job_subject'])
                .experiment(job['job_exp_label'])
                .id()
            )

            res2 = wa.launch_cs_command(
                xnat_interface,
                project,
                job['job_subject'],
                job['job_exp_label'],
                exp_id,
                workflow_id,
                xnat_command_id,
                xnat_command_wrapper_id,
                verbose=True
            )
            if not res2:
                raise ConnectionError("Launching container job failed")

        if n % 10 == 0:
            print(f'Done {n} out of {num_sessions} ({n*100/num_sessions:.1f}%)')


Written /workspace/mmilchenko/NSCLC_RADIOMICS/jobs/nsclc-segmentation-codebase-20260317_LUNG1-097_02-26-2006-StudyID-NA-21320/job.sh
2026-04-23 18:54:59,987 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
2026-04-23 18:54:59,987 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
Sending job bundle to XNAT resource
Submitting job to Container Service
Request body:  {'PROJECT': 'NSCLC_RADIOMICS', 'SUBJECT': 'LUNG1-097', 'EXPERIMENT': '02-26-2006-StudyID-NA-21320', 'WORKFLOW_ID': 'nsclc-segmentation-codebase-20260317', 'MICROENV': 'NONE', 'REPO_GIT': 'NONE', 'session': 'TAP03_E00435'}
Request url:  https://tap.embarklabs.ai/xapi/projects/NSCLC_RADIOMICS/commands/21/wrappers/xnat-ai-workflow-gevaert/launch
Status: 200
Reason: 
URL: https://tap.embarklabs.ai/xapi/projects/NSCLC_RADIOMICS/commands/21/wrappers/xnat-ai-workflow-gevaert/launch
Headers:
 {'Date': 'Thu, 23 Apr 2026 18:55:01 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'k

## 6. Check for missing outputs

Run the next cell after jobs finish to identify sessions without expected output.


In [ ]:
n = 0
n_failed = 0
sessions_failed = []

for scan in scans:
    n = n + 1
    if n < start_pos or n > end_pos:
        continue

    job = wa.populate_job_fields(env_type, global_vars, workflow_id, scan)
    expected_mask = job['job_workdir'] / 'lesion_mask.nii.gz'
    expected_qc = job['job_workdir'] / 'qc.png'

    if not expected_mask.exists() or not expected_qc.exists():
        n_failed += 1
        sessions_failed.append({
            'Subject': job['job_subject'],
            'Experiment': job['job_exp_label'],
            'mask_exists': expected_mask.exists(),
            'qc_exists': expected_qc.exists()
        })

print(f'Failed sessions: {n_failed}')
sessions_failed[:10]
